In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt
import matplotlib.pyplot as plt


Fourier Transform in Numpy

###### First we will see how to find Fourier Transform using Numpy. Numpy has an FFT package to do this. np.fft.fft2() provides us the frequency transform which will be a complex array. Its first argument is the input image, which is grayscale.

Step 1: Load and Convert Image to Grayscale

In [ ]:
img = cv2.imread('meowo.jpg',0)
f = np.fft.fft2(img)
fshift = np.fft.fftshift(f)
magnitude_spectrum = 20*np.log(np.abs(fshift))

plt.subplot(121),plt.imshow(img, cmap = 'gray')
plt.title('Input Image'), plt.xticks([]), plt.yticks([])
plt.subplot(122),plt.imshow(magnitude_spectrum, cmap = 'gray')
plt.title('Magnitude Spectrum'), plt.xticks([]), plt.yticks([])
plt.show()

Step 2: Compute the 2D Discrete Fourier Transform (DFT)


Now you can do some operations in frequency domain, like high pass filtering and reconstruct the image, ie find inverse DFT. 

In [ ]:
rows, cols = img.shape
crow,ccol = rows/2 , cols/2
#fshift[crow-30:crow+30, ccol-30:ccol+30] = 0
f_ishift = np.fft.ifftshift(fshift)
img_back = np.fft.ifft2(f_ishift)
img_back = np.abs(img_back)

plt.subplot(131),plt.imshow(img, cmap = 'gray')
plt.title('Input Image'), plt.xticks([]), plt.yticks([])
plt.subplot(132),plt.imshow(img_back, cmap = 'gray')
plt.title('Image after HPF'), plt.xticks([]), plt.yticks([])
plt.subplot(133),plt.imshow(img_back)
plt.title('Result in JET'), plt.xticks([]), plt.yticks([])

plt.show()

Fourier Transform in OpenCV


###### OpenCV provides the functions cv2.dft() and cv2.idft() for this. It returns the same result as previous, but with two channels. First channel will have the real part of the result and second channel will have the imaginary part of the result.

In [ ]:
import numpy as np
import cv2
from matplotlib import pyplot as plt

img = cv2.imread('meowo.jpg',0)

dft = cv2.dft(np.float32(img),flags = cv2.DFT_COMPLEX_OUTPUT)
dft_shift = np.fft.fftshift(dft)

magnitude_spectrum = 20*np.log(cv2.magnitude(dft_shift[:,:,0],dft_shift[:,:,1]))

plt.subplot(121),plt.imshow(img, cmap = 'gray')
plt.title('Input Image'), plt.xticks([]), plt.yticks([])
plt.subplot(122),plt.imshow(magnitude_spectrum, cmap = 'gray')
plt.title('Magnitude Spectrum'), plt.xticks([]), plt.yticks([])
plt.show()

Step 3: Apply Frequency Domain Gaussian Filters

Inverse DFT - we will see how to remove high frequency contents in the image, ie we apply LPF to image. It actually blurs the image.

In [ ]:
def create_gaussian_filter(img, cutoff, highpass=False):
    rows, cols = img.shape
    crow, ccol = rows // 2, cols // 2  # center of the image
    
    # Generate coordinate grids for filter
    x = np.arange(0, cols)
    y = np.arange(0, rows)
    x, y = np.meshgrid(x, y)
    
    # Gaussian filter creation
    gaussian = np.exp(-((x - ccol)**2 + (y - crow)**2) / (2 * (cutoff**2)))
    
    # High-pass filter: subtract low-pass from 1
    if highpass:
        gaussian = 1 - gaussian
        print("This is a High-Pass filter.")
    else:
        print("This is a Low-Pass filter.")
    # Create a mask with the Gaussian filter applied
    return gaussian

# Load image (ensure you replace this with your actual image)
img = cv2.imread('meowo.jpg', 0)  # Read as grayscale

# Perform DFT and shift zero frequency component to center
dft = np.fft.fft2(img)
dft_shift = np.fft.fftshift(dft)

# Create Gaussian filter (low-pass)
cutoff = 50  # Cutoff frequency
gaussian_filter = create_gaussian_filter(img, cutoff, highpass=False)

# Apply Gaussian filter in frequency domain
fshift = dft_shift * gaussian_filter

# Inverse DFT to get back to spatial domain
f_ishift = np.fft.ifftshift(fshift)
img_back = np.fft.ifft2(f_ishift)
img_back = np.abs(img_back)  # Get the magnitude

# Plot the original and filtered images
plt.subplot(121), plt.imshow(img, cmap='gray')
plt.title('Input Image'), plt.xticks([]), plt.yticks([])
plt.subplot(122), plt.imshow(img_back, cmap='gray')
plt.title('Applied Gaussian Filter (Frequency Domain)'), plt.xticks([]), plt.yticks([])

plt.show()

Applies a filter mask in the frequency domain by multiplying the DFT with the filter mask element-wise.
    Also visualizes the magnitude spectrum of the original and filtered DFT.

In [ ]:

filter_mask = create_gaussian_filter(img.shape, cutoff=30, highpass=True)

# filter function
def apply_filter_in_frequency_domain(dft, filter_mask):
    filtered_dft = np.zeros_like(dft)
    filtered_dft[:, :, 0] = dft[:, :, 0] * filter_mask
    filtered_dft[:, :, 1] = dft[:, :, 1] * filter_mask

    dft_shift = np.fft.fftshift(filtered_dft)
    magnitude_spectrum = cv2.magnitude(dft_shift[:, :, 0], dft_shift[:, :, 1])
    magnitude_spectrum = np.log1p(magnitude_spectrum)

    # Show result
    plt.figure(figsize=(6, 6))
    plt.imshow(magnitude_spectrum, cmap='gray')
    plt.title('Filtered DFT Magnitude Spectrum')
    plt.axis('off')
    plt.show()

    return filtered_dft

filtered_dft = apply_filter_in_frequency_domain(dft, filter_mask)

Next Step is INVERSE DFT

BUT FIRST LET PERFORM OTHER STEPS LIKE

Performance Optimization of DFT


In [ ]:
img = cv2.imread('meowo.jpg',0)
rows,cols = img.shape
print(rows,cols)
nrows = cv2.getOptimalDFTSize(rows)
ncols = cv2.getOptimalDFTSize(cols)
print (nrows, ncols)

nimg = np.zeros((nrows,ncols))
nimg[:rows,:cols] = img


We calculate the DFT performance comparison of Numpy function:

In [ ]:
#%timeit fft1 = np.fft.fft2(img)10 loops, best of 3: 40.9 ms per loop
#%timeit fft2 = np.fft.fft2(img,[nrows,ncols])100 loops, best of 3: 10.4 ms per loop
#%timeit dft1= cv2.dft(np.float32(img),flags=cv2.DFT_COMPLEX_OUTPUT)100 loops, best of 3: 13.5 ms per loop
#%timeit dft2= cv2.dft(np.float32(nimg),flags=cv2.DFT_COMPLEX_OUTPUT)100 loops, best of 3: 3.11 ms per loop

FFT Magnitude Spectrum of Different Filters

In [ ]:
# simple averaging filter without scaling parameter
mean_filter = np.ones((3,3))

# creating a guassian filter
x = cv2.getGaussianKernel(5,10)
gaussian = x*x.T

# different edge detecting filters
# scharr in x-direction
scharr = np.array([[-3, 0, 3],
                   [-10,0,10],
                   [-3, 0, 3]])
# sobel in x direction
sobel_x= np.array([[-1, 0, 1],
                   [-2, 0, 2],
                   [-1, 0, 1]])
# sobel in y direction
sobel_y= np.array([[-1,-2,-1],
                   [0, 0, 0],
                   [1, 2, 1]])
# laplacian
laplacian=np.array([[0, 1, 0],
                    [1,-4, 1],
                    [0, 1, 0]])

filters = [mean_filter, gaussian, laplacian, sobel_x, sobel_y, scharr]
filter_names = ['mean_filter', 'gaussian','laplacian', 'sobel_x', \
                'sobel_y', 'scharr_x']
# Compute FFT and Magnitude Spectrums
fft_filters = [np.fft.fft2(f, s=(32,32)) for f in filters]  # Padding for nicer view
fft_shifted = [np.fft.fftshift(f) for f in fft_filters]
mag_spectrums = [np.log(np.abs(f) + 1) for f in fft_shifted]

plt.figure(figsize=(14, 7))
for i in range(6):
    plt.subplot(2, 3, i+1)
    plt.imshow(mag_spectrums[i], cmap='jet')
    plt.title(filter_names[i], fontsize=12)
    plt.axis('off')

plt.suptitle('FFT Magnitude Spectrum of Different Filters ', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95]) 
plt.show()

Now we will Write a function compute_idft(filtered_dft) that applies reverse frequency to the image making it shift

In [ ]:
def compute_idft(filtered_dft):
    f_ishift = np.fft.ifftshift(filtered_dft)
    img_back = cv2.idft(f_ishift)
    img_back = cv2.magnitude(img_back[:, :, 0], img_back[:, :, 1])

    
    img_back_magnitude = cv2.normalize(img_back, None, 0, 255, cv2.NORM_MINMAX)
    img_back_magnitude = img_back_magnitude.astype(np.uint8)

    # Plot the reconstructed image
    plt.figure(figsize=(6, 5))
    plt.imshow(img_back_magnitude, cmap='gray')
    plt.title('Resulting Spatial of (IDFT)')
    plt.axis('off')
    plt.show()

    return img_back_magnitude

reconstructed_img = compute_idft(filtered_dft)

Display Results of the Four images using matplotlib.pyplot

In [ ]:
def load_grayscale_image(path):
    return cv2.imread(path, cv2.IMREAD_GRAYSCALE)

def compute_dft(img):
    # Compute DFT and shift zero-frequency component to the center
    dft = cv2.dft(np.float32(img), flags=cv2.DFT_COMPLEX_OUTPUT)
    dft_shift = np.fft.fftshift(dft)
    
    # Compute the magnitude spectrum (log scale)
    magnitude_spectrum = cv2.magnitude(dft_shift[:, :, 0], dft_shift[:, :, 1])
    magnitude_spectrum = np.log(magnitude_spectrum + 1)
    
    return dft_shift, magnitude_spectrum

def compute_idft(filtered_dft):
    f_ishift = np.fft.ifftshift(filtered_dft)
    img_back = cv2.idft(f_ishift)
    img_back = cv2.magnitude(img_back[:, :, 0], img_back[:, :, 1])

    img_back_magnitude = cv2.normalize(img_back, None, 0, 255, cv2.NORM_MINMAX)
    img_back_magnitude = img_back_magnitude.astype(np.uint8)

    return img_back_magnitude

def create_gaussian_filter(shape, cutoff, highpass=False):
    rows, cols = shape
    crow, ccol = rows // 2, cols // 2

    # Create a Gaussian filter
    x = np.fft.fftfreq(cols, 1 / cols)
    y = np.fft.fftfreq(rows, 1 / rows)
    X, Y = np.meshgrid(x, y)
    dist = np.sqrt(X**2 + Y**2)

    if highpass:
        filter = 1 - np.exp(-dist**2 / (2 * (cutoff)**2))
    else:
        filter = np.exp(-dist**2 / (2 * (cutoff)**2))

    filter = np.fft.fftshift(filter)  # Shift to center
    return filter

def apply_filter_in_frequency_domain(dft_shift, filter):
    # Apply the filter to the frequency domain
    return dft_shift * filter[:, :, np.newaxis]

def plot_results(original_img, magnitude_spectrum, lp_image, hp_image):
    plt.figure(figsize=(12, 8))

    plt.subplot(2, 2, 1)
    plt.imshow(original_img, cmap='gray')
    plt.title('Original Image')
    plt.axis('off')

    plt.subplot(2, 2, 2)
    plt.imshow(magnitude_spectrum, cmap='gray')
    plt.title('Magnitude Spectrum of DFT')
    plt.axis('off')

    plt.subplot(2, 2, 3)
    plt.imshow(lp_image, cmap='gray')
    plt.title('Low-Pass Filtered Image')
    plt.axis('off')

    plt.subplot(2, 2, 4)
    plt.imshow(hp_image, cmap='gray')
    plt.title('High-Pass Filtered Image')
    plt.axis('off')

    plt.show()

if __name__ == '__main__':
    path = 'meowo.jpg'
    img = load_grayscale_image(path)

    dft_shift, magnitude_spectrum = compute_dft(img)

    lp_filter = create_gaussian_filter(img.shape, cutoff=30, highpass=False)
    hp_filter = create_gaussian_filter(img.shape, cutoff=30, highpass=True)

    lp_dft = apply_filter_in_frequency_domain(dft_shift, lp_filter)
    hp_dft = apply_filter_in_frequency_domain(dft_shift, hp_filter)

    lp_image = compute_idft(lp_dft)
    hp_image = compute_idft(hp_dft)

    plot_results(img, magnitude_spectrum, lp_image, hp_image)
